# Student Placement Prediction using Microsoft Azure Machine Learning
## Academic Internship Project - Azure AI-900 (Azure AI Fundamentals)

This notebook implements the complete end-to-end Machine Learning pipeline for binary classification:
1. **Exploratory Data Analysis (EDA)**
2. **Data Cleaning & Missing Value Imputation**
3. **Feature Scaling & Preprocessing Pipeline**
4. **Model Training & Comparison (Logistic Regression, Decision Tree, Random Forest)**
5. **Model Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, 5-Fold Cross Validation)**
6. **Model Selection & Artifact Serialization (`model.pkl`, `metrics.json`)**

### Step 1: Import Required Libraries

In [1]:
import os
import json
import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    classification_report
)

print(f"Scikit-Learn Version: {sklearn.__version__}")

### Step 2: Load Placement Dataset and Inspect Properties

In [2]:
# Load dataset from data folder
data_path = os.path.join("..", "data", "placement_data.csv")
df = pd.read_csv(data_path)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Step 3: Exploratory Data Analysis (EDA) & Summary Statistics

In [3]:
# Check numerical summary
print("--- Summary Statistics ---")
display(df.describe().round(2))

# Check missing values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

### Step 4: Visualizing Target Class Balance and Feature Correlations

In [4]:
feature_cols = ['cgpa', 'internships', 'projects', 'communication_skills', 'aptitude_score', 'backlogs', 'certifications']
target_col = 'placed'

# 1. Class Balance Bar Plot
plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df, palette=["#e74c3c", "#2ecc71"])
plt.title("Target Distribution (Placed)", fontsize=13, fontweight='bold')
plt.show()

# 2. Correlation Matrix
plt.figure(figsize=(8, 6))
corr_df = df[feature_cols].copy()
corr_df['placed_num'] = df[target_col].map({'Yes': 1, 'No': 0})
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Matrix", fontsize=13, fontweight='bold')
plt.show()

### Step 5: Train-Test Split (70% Train, 30% Test with Stratification)

In [5]:
X = df[feature_cols]
y = df[target_col].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f"Training set samples: {X_train.shape[0]}")
print(f"Testing set samples:  {X_test.shape[0]}")

### Step 6: Model Training & 5-Fold Cross Validation
We encapsulate median imputation (`SimpleImputer`), standard scaling (`StandardScaler`), and the classifier inside a Scikit-Learn `Pipeline` to prevent data leakage.

In [6]:
candidate_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=500),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_leaf=10),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6, min_samples_leaf=5)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
trained_pipelines = {}

for name, model in candidate_models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    
    # 5-fold CV
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'CV_F1_Mean': cv_scores.mean()
    }

results_df = pd.DataFrame(results).T.round(4)
print("=== Model Performance Comparison ===")
display(results_df)

### Step 7: Confusion Matrix and ROC Curve for the Best Performing Model

In [7]:
best_model_name = results_df['F1-Score'].idxmax()
best_pipe = trained_pipelines[best_model_name]
print(f"Best Selected Model: {best_model_name}")

# Confusion Matrix
y_pred_best = best_pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.title(f"Confusion Matrix ({best_model_name})", fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ROC Curves
plt.figure(figsize=(7, 5))
for name, pipe in trained_pipelines.items():
    probs = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, probs):.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.title("Receiver Operating Characteristic (ROC) Curves", fontweight='bold')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

### Step 8: Save Pipeline to Disk for Deployment (`model.pkl`)

In [8]:
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)
joblib.dump(best_pipe, os.path.join(models_dir, "model.pkl"))
print("Best model pipeline successfully serialized to models/model.pkl")